## 1- Standards import 

In [1]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random


## 2- Reproducability 
- Fixed  random seeds so sampling and simulation outputs are repeatable

In [2]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 3- Simulation parameters

since we have decided to simplify the system to have only Acust Stroke unit here are left 

In [3]:
SIM_HOURS = 30          # simulation run length (hours)
MEAN_IAT_HOURS = 4      # mean interarrival time (hours) - placeholder
ASU_BEDS = 5            # number of acute beds (placeholder)

## 4- Enviroment setup
- needs to be run every time we need to execute the simulation 

In [ ]:
env = simpy.Environment()
beds = simpy.Resource(env, capacity=ASU_BEDS)

In [ ]:
ROUTING_ACUTE = {
    "stroke": {"rehab": 0.24, "esd": 0.13, "other": 0.63},
    "tia":    {"rehab": 0.01, "esd": 0.01, "other": 0.98},
    "neuro":  {"rehab": 0.11, "esd": 0.05, "other": 0.84},  # complex-neurological
    "other":  {"rehab": 0.05, "esd": 0.10, "other": 0.85},
}

def choose_route(routing_dict, patient_type):
    dests = list(routing_dict[patient_type].keys())
    probs = list(routing_dict[patient_type].values())
    return random.choices(dests, weights=probs, k=1)[0]

## 5- LOS sampling helper (lognormal from mean+SD)
- The appendix says acute LOS is lognormal and provides mean and SD (in days).
NumPy needs lognormal parameters (mu, sigma), so we convert.

In [5]:
def sample_lognormal_from_mean_sd(mean, sd, rng=np.random):
    """
    Sample from a lognormal distribution parameterised by mean and sd
    on the ORIGINAL scale (days).
    """
    sigma2 = np.log(1 + (sd**2) / (mean**2))
    sigma = np.sqrt(sigma2)
    mu = np.log(mean) - 0.5 * sigma2
    return rng.lognormal(mean=mu, sigma=sigma)

## 6- Acute LOS parameters (days) from the appendix

In [ ]:
ACUTE_LOS_DAYS = {
    "stroke": (7.4, 8.6),   # Strokes – No ESD
    "tia":    (1.8, 2.3),   # TIA
    "neuro":  (4.0, 5.0),   # Complex-neurological
    "other":  (3.8, 5.2)    # Other
}

## 7- Patient process (requests bed + stays LOS)
- This is the core process.
Patients arrive → request a bed → wait if full → sample LOS → stay → leave.

In [ ]:
def patient(env, name, patient_type, beds):
    arrival_time = env.now
    print(f"{name} ({patient_type}) arrives at {arrival_time:.2f}h")

    with beds.request() as req:
        yield req
        bed_time = env.now
        wait = bed_time - arrival_time
        print(f"{name} got a bed at {bed_time:.2f}h (wait {wait:.2f}h)")

        # Lognormal LOS from appendix (days -> hours)
        mean_d, sd_d = ACUTE_LOS_DAYS[patient_type]
        los_days = sample_lognormal_from_mean_sd(mean_d, sd_d)
        los_hours = los_days * 24

        yield env.timeout(los_hours)

        print(f"{name} leaves at {env.now:.2f}h (LOS {los_days:.2f} days = {los_hours:.2f}h)")
    
    # ROUTE OUT OF ACUTE
    dest = choose_route(ROUTING_ACUTE, patient_type)
    print(f"{name} routes from ACUTE -> {dest.upper()} at {env.now:.2f}h")

## 8- Arrival generator (creates patients over time)
- Generates interarrival times and patient types, then starts patient processes.

In [ ]:
def arrival_process(env, beds):
    i = 0
    while True:
        # Exponential interarrival times (Poisson arrivals) - placeholder parameter
        iat = random.expovariate(1 / MEAN_IAT_HOURS)
        yield env.timeout(iat)

        i += 1
        patient_type = random.choices(
            ["stroke", "tia", "neuro", "other"],
            weights=[0.5, 0.2, 0.15, 0.15]   # placeholder mix
        )[0]

        env.process(patient(env, f"Patient {i}", patient_type, beds))

## 9- Run the simulation 

In [ ]:
env = simpy.Environment()
beds = simpy.Resource(env, capacity=ASU_BEDS)

env.process(arrival_process(env, beds))
env.run(until=SIM_HOURS)

Patient 1 (stroke) arrives at 4.08h
Patient 1 got a bed at 4.08h (wait 0.00h)
Patient 2 (stroke) arrives at 5.37h
Patient 2 got a bed at 5.37h (wait 0.00h)
Patient 3 (tia) arrives at 10.70h
Patient 3 got a bed at 10.70h (wait 0.00h)
Patient 4 (stroke) arrives at 19.61h
Patient 4 got a bed at 19.61h (wait 0.00h)
Patient 5 (stroke) arrives at 21.80h
Patient 5 got a bed at 21.80h (wait 0.00h)
Patient 6 (tia) arrives at 22.79h
Patient 7 (stroke) arrives at 22.90h
Patient 8 (tia) arrives at 27.09h
Patient 9 (tia) arrives at 28.09h


## 10- Sampling validation (your “sampling” deliverable)
- Show that your sampling function produces mean/SD close to the appendix values.

In [10]:
N = 50000
for t in ["stroke", "tia", "neuro", "other"]:
    m, s = ACUTE_LOS_DAYS[t]
    samples = np.array([sample_lognormal_from_mean_sd(m, s) for _ in range(N)])
    print(
        f"{t:6s} target mean={m:.2f}, sd={s:.2f} | "
        f"sample mean={samples.mean():.2f}, sd={samples.std(ddof=1):.2f}"
    )

stroke target mean=7.40, sd=8.60 | sample mean=7.39, sd=8.52
tia    target mean=1.80, sd=2.30 | sample mean=1.81, sd=2.29
neuro  target mean=4.00, sd=5.00 | sample mean=4.01, sd=4.97
other  target mean=3.80, sd=5.20 | sample mean=3.78, sd=5.01
